# Topic 25 — Classical NLP Classification
### ⭐ Directly relevant to your cyberbullying paper. This notebook builds a complete baseline pipeline.

Everything from Topics 6, 9, 11, 12, 13, 14, 17, 21, 22, 23, 24 comes together here:

```text
raw text -> preprocessing -> TF-IDF -> { LogReg | Naive Bayes | SVM | Random Forest } -> evaluation
```

This is the standard "classical ML baseline" structure for a text classification paper — you'd
build exactly this, swap in your real dataset, and report the results table it produces.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)

rng = np.random.default_rng(0)

## 1. A slightly larger synthetic labeled dataset

In your real project this cell is replaced by `pd.read_csv("your_dataset.csv")` (Topic 2).
Everything downstream works identically once you have real `text`/`label` columns.

In [ ]:
bullying_examples = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "you should just disappear", "you are pathetic",
    "everyone thinks you're an idiot", "just go away nobody likes you",
    "you're so ugly and useless", "why do you even exist",
    "you deserve to be alone", "stop talking you sound stupid",
]
not_bullying_examples = [
    "great job today team", "have a wonderful day", "nice work everyone",
    "thanks for your help", "well done on the project", "excellent effort today",
    "looking forward to the weekend", "congratulations on your achievement",
    "the weather is nice today", "let's grab coffee sometime",
    "i really appreciate your feedback", "the meeting went smoothly",
]

texts = bullying_examples + not_bullying_examples
labels = np.array([1]*len(bullying_examples) + [0]*len(not_bullying_examples))

df = pd.DataFrame({"text": texts, "label": labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle
print(df.head())
print("\nclass balance:", df["label"].value_counts().to_dict())

## 2. Preprocessing (reusing Topic 21)

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    return text.strip()

df["clean_text"] = df["text"].apply(preprocess_text)
print(df[["text", "clean_text"]].head())

## 3. Train/test split (stratified — Topic 6)

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)
print("train size:", len(X_train_txt), " test size:", len(X_test_txt))
print("train class balance:", np.bincount(y_train))
print("test class balance:", np.bincount(y_test))

## 4. Build 4 pipelines: TF-IDF + each classifier

Each is a `Pipeline` (Topic 17) so vectorizer fitting stays safely inside `.fit()`, never leaking
into test data.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "SVM (Linear)": LinearSVC(class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
}

pipelines = {
    name: Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
        ("clf", model),
    ])
    for name, model in models.items()
}

for name, pipe in pipelines.items():
    pipe.fit(X_train_txt, y_train)
print("all models trained.")

## 5. Evaluate every model (Topic 9's metrics)

In [ ]:
results = []
for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test_txt)
    results.append({
        "model": name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(results_df.round(3))
# This exact table (precision/recall/F1 per model) is the kind of results table you'd put
# directly in your paper's experiments section.

In [ ]:
results_df.set_index("model")[["precision", "recall", "f1"]].plot(kind="bar", figsize=(8, 5))
plt.title("Classical NLP model comparison")
plt.ylabel("score")
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 6. Confusion matrices for every model

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, pipe) in zip(axes, pipelines.items()):
    y_pred = pipe.predict(X_test_txt)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["not_bully", "bully"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 7. More reliable comparison with cross-validation (Topic 6)

A single train/test split on a small dataset can be noisy. Cross-validated F1 gives a steadier estimate.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, df["clean_text"], df["label"], cv=cv, scoring="f1")
    print(f"{name:<22} mean F1 = {scores.mean():.3f}  (+/- {scores.std():.3f})")

## 8. Inspecting which words drive predictions (a first taste of interpretability — Topic 40)

In [ ]:
logreg_pipe = pipelines["Logistic Regression"]
tfidf_step = logreg_pipe.named_steps["tfidf"]
clf_step = logreg_pipe.named_steps["clf"]

feature_names = tfidf_step.get_feature_names_out()
coefficients = clf_step.coef_[0]

top_bullying_idx = np.argsort(coefficients)[::-1][:10]
top_not_bullying_idx = np.argsort(coefficients)[:10]

print("top words/phrases pushing toward 'bullying':")
for i in top_bullying_idx:
    print(f"  {feature_names[i]:<20} weight={coefficients[i]:.3f}")

print("\ntop words/phrases pushing toward 'not bullying':")
for i in top_not_bullying_idx:
    print(f"  {feature_names[i]:<20} weight={coefficients[i]:.3f}")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add 10 more examples of your own to bullying_examples/not_bullying_examples and re-run
#    everything -- watch how metrics change with more data.
# 2. Swap TfidfVectorizer for CountVectorizer in the pipelines and compare the results table --
#    does TF-IDF actually help here, or is it close?
# 3. Try ngram_range=(1,1) vs (1,2) vs (1,3) in the TfidfVectorizer step for just the SVM
#    pipeline and compare cross-validated F1.
# 4. Once you load YOUR real cyberbullying dataset in Topic-2 style (pd.read_csv), swap it in for
#    `df` here (keep columns named "clean_text" and "label") -- this entire notebook should run
#    on your real data with minimal changes.

---
### Next up: **Topic 26 — Word Embeddings** (moving beyond sparse TF-IDF to dense vectors).

Say "next" when you're ready.